#### Bad Code first
I asked Gemini (model fast, February 2026) to write a Python script to chunk the PDFs into smaller chunks.  
This is the code it generated:  
Note that the code is working, but it is not readable and not maintainable.  
I had to ask Gemini to write it again, which you can see in the next file.  

In [ ]:
import os
import json
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- CONFIGURATION ---
INPUT_FOLDER = "./medcare_docs"  # Folder containing your 10 PDFs
OUTPUT_FILE = "medcare_knowledge_base.jsonl"

# The Splitter: 
# We use a 1000 character limit with 150 char overlap.
# This ensures that if a medical warning is at the end of a chunk, 
# it's repeated at the start of the next for context.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def process_medcare_pdfs():
    chunk_count = 0
    
    # Ensure the output file is fresh
    if os.path.exists(OUTPUT_FILE):
        os.remove(OUTPUT_FILE)

    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f:
        # Loop through all files in your laptop's folder
        for filename in os.listdir(INPUT_FOLDER):
            if filename.lower().endswith(".pdf"):
                file_path = os.path.join(INPUT_FOLDER, filename)
                print(f"🧬 Processing: {filename}...")

                # Open PDF with PyMuPDF
                doc = fitz.open(file_path)
                
                for page_num, page in enumerate(doc):
                    # Extract text from the current page
                    page_text = page.get_text("text")
                    
                    # Generate chunks for this specific page
                    chunks = text_splitter.split_text(page_text)
                    
                    for i, chunk_content in enumerate(chunks):
                        # Create the structured dictionary
                        entry = {
                            "chunk_id": f"{filename}_{page_num}_{i}",
                            "text": chunk_content.strip(),
                            "metadata": {
                                "source_file": filename,
                                "page_number": page_num + 1,
                                "org": "Medcare",
                                "content_type": "Medical Record/Policy"
                            }
                        }
                        
                        # Write to JSONL (one JSON object per line)
                        f.write(json.dumps(entry) + '\n')
                        chunk_count += 1
                
                doc.close()

    print(f"✅ Success! Created {chunk_count} chunks in {OUTPUT_FILE}")

if __name__ == "__main__":
    # Create the folder if it doesn't exist for the user
    if not os.path.exists(INPUT_FOLDER):
        os.makedirs(INPUT_FOLDER)
        print(f"Put your 10 PDFs in the '{INPUT_FOLDER}' folder and run again.")
    else:
        process_medcare_pdfs()